In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

rng = np.random.default_rng(42)

def metrics(estimates, theta):
    # estimates: (samples, #estimators)
    # theta: (1,)
    obciazenie = estimates.mean(axis=0) - theta   # (#estimators,) - (1,) = (#estimators)
    wariancja = estimates.var(axis=0)             # (#estimators)
    mse = ((estimates - theta) ** 2).mean(axis=0) # (#estimators)
    return np.column_stack((obciazenie, wariancja, mse))  # (#estimators, 3)


def results(metric_values, names) -> None:
    """Wyświetla tabelę miar jakości estymatorów."""
    print(f"{'estymator':<22} {'obciazenie':>12} {'wariancja':>12} {'mse':>12}")
    for name, row in zip(names, metric_values):
        print(f"{name:<22} {row[0]:12.5g} {row[1]:12.5g} {row[2]:12.5g}")


def boxes(ax, estimates, names, theta, title) -> None:
    """Rysuje wykres pudełkowy"""
    ax.boxplot(estimates, tick_labels=names, showfliers=False)
    ax.axhline(theta, color="tab:red", linestyle=":")
    ax.set_title(title)
    ax.tick_params(axis="x", labelrotation=25)
    ax.set_ylabel("wartosc estymatora")
samples = 10000
cases = [(0, 1), (0, 2), (4, 1), (4, 2)]  # (theta, sigma)


In [ ]:
# zadanie 2
n = 50
names = ["srednia", "mediana"]

for rozklad in ["normalny", "logistyczny", "cauchy"]:
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    for ax, (theta, sigma) in zip(axes.flat, cases):
        if rozklad == "normalny":
            x = rng.normal(theta, sigma, size=(samples, n))
        elif rozklad == "logistyczny":
            x = rng.logistic(theta, sigma, size=(samples, n))
        else:
            x = theta + sigma * rng.standard_cauchy(size=(samples, n))
        srednia = x.mean(axis=1)
        mediana = np.median(x, axis=1)
        estimates = np.column_stack([srednia, mediana])
        print(f"\n{rozklad}, theta = {theta}, sigma = {sigma}")
        metric_values = metrics(estimates, theta)
        results(metric_values, names)
        boxes(ax, estimates, names, theta, f"theta = {theta}, sigma = {sigma}")
    fig.suptitle(f"zadanie 2, rozklad {rozklad}")
    fig.tight_layout()
    plt.show()


In [ ]:
# zadanie 3
n = 50
ks = np.arange(10, 101, 10)
base = rng.normal(size=(samples, n - 1))
results_3 = []

for k in ks:
    x = np.column_stack([base, np.full(samples, k)])
    srednia = x.mean(axis=1)
    mediana = np.median(x, axis=1)
    estimates = np.column_stack([srednia, mediana])
    print(f"\nk = {k}")
    metric_values = metrics(estimates, 0)
    results(metric_values, ["srednia", "mediana"])
    results_3.append(metric_values)

results_3 = np.array(results_3)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for j, (ax, title) in enumerate(zip(axes, ["obciazenie", "wariancja", "mse"])):
    for i, name in enumerate(["srednia", "mediana"]):
        ax.plot(ks, results_3[:, i, j], marker="o", label=name)
    wartosc_teoretyczna = [ks / n, np.full(len(ks), (n - 1) / n ** 2), (n - 1) / n ** 2 + (ks / n) ** 2][j]
    ax.plot(ks, wartosc_teoretyczna, linestyle=":", color="black", label="srednia, teoria")
    ax.set(xlabel="k", ylabel=title, title=title)
    ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# zadanie 4
n = 50
names = ["wariancja proby", "wokol mediany", "mediana kwadratow"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for ax, (mu, theta) in zip(axes.flat, [(0, 1), (0, 4), (4, 1), (4, 4)]):
    x = rng.normal(mu, np.sqrt(theta), size=(samples, n))
    srednia = x.mean(axis=1, keepdims=True)
    mediana = np.median(x, axis=1, keepdims=True)
    wariancja_proby = np.sum((x - srednia) ** 2, axis=1) / (n - 1)
    wariancja_wokol_mediany = np.sum((x - mediana) ** 2, axis=1) / (n - 1)
    mediana_kwadratow = np.median((x - srednia) ** 2, axis=1)

    estimates = np.column_stack([
        wariancja_proby, wariancja_wokol_mediany, mediana_kwadratow,
    ])
    print(f"\nmu = {mu}, theta = {theta}")
    metric_values = metrics(estimates, theta)
    results(metric_values, names)
    boxes(ax, estimates, names, theta, f"mu = {mu}, theta = {theta}")
fig.suptitle("zadanie 4")
fig.tight_layout()
plt.show()
